In [ ]:
import os
import sys
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from tqdm import tqdm
from datetime import datetime

sys.path.append("../utils")

import config
import data_utils
import planet_api
import planet_utils

In [ ]:
PLANET_API_KEY = planet_api.key
# planet = planet_utils.PlanetAPI(PLANET_API_KEY, planet_utils.basemap_url)
# mosaic_names = planet.get_mosaics()

# mosaic_names = [name for name in mosaic_names if "monthly" in name]
# mosaic_names

In [46]:
PLANET_API_KEY = planet_api.key

planet = planet_utils.PlanetBasemapsAPI(api_key=PLANET_API_KEY)
print(f"Authentication status: {planet.check_authentication_status()}")


mosaics = planet.get_mosaics(search_str="global_monthly")

# Define your start and end dates
start_date = datetime(2019, 1, 1)
end_date = datetime(2023, 12, 31)  # Include all of January 2024

# Helper function to parse year and month from mosaic name
def extract_date(mosaic_name: str) -> datetime:
    """Extract datetime object from a mosaic name string."""
    parts = mosaic_name.split('_')
    year = int(parts[2])
    month = int(parts[3])
    return datetime(year, month, 1)

# Filter mosaics
mosaics = [
    mosaic for mosaic in mosaics
    if start_date <= extract_date(mosaic) <= end_date
]

for mosaic in mosaics:
    print(mosaic)

    # mosaic = mosaics[-1]

    planet.set_mosaic(mosaic)

    mosaic_dir = os.path.join(
        config.basemap_dir,
        mosaic,
    )

    file_dir = os.path.join(mosaic_dir, "quad_ids")
    download_dir = os.path.join(mosaic_dir, "basemap_quads")

    os.makedirs(file_dir, exist_ok=True)
    os.makedirs(download_dir, exist_ok=True)

    quad_fn = os.path.join(file_dir, "quad_ids.geojson")
    # grid_fn = os.path.join(file_dir, "grid_coordinates.parquet")

    counties = gpd.read_file(
        os.path.join(config.data_dir, "ca_counties", "CA_Counties.shp")
    )
    counties = counties.to_crs(config.geodetic_crs)
    sb_county = counties.loc[counties["NAME"] == "Santa Barbara"]
    bbox = [-125, 34.25, -119.0, 38.0]
    sb_county = sb_county.clip(bbox)

    bbox_aoi = sb_county.geometry.total_bounds

    quads = planet.get_items(bbox_aoi)

    quad_df = planet.convert_items_to_geodataframe(quads)
    quad_df = quad_df.drop_duplicates(subset="id")

    quad_df = quad_df.sjoin(sb_county, predicate="intersects")
    quad_df.drop(columns=["index_right"], inplace=True)

    filtered_ids = set(quad_df["id"].unique())

    if not os.path.exists(quad_fn):
        quad_df.to_file(quad_fn, driver="GeoJSON")

    quads = [q for q in quads if q["id"] in filtered_ids]

    planet.download_quads(
        quads,
        directory=download_dir,
        overwrite=True,
        log_interval=10,
    )

Authentication status: Authentication successful.
global_monthly_2019_01_mosaic
2025-04-27 22:00:12,225 - INFO - Created 43 records
2025-04-27 22:00:25,104 - INFO - Progress: 10/43 quads processed (10 downloaded, 0 skipped)
2025-04-27 22:00:38,493 - INFO - Progress: 20/43 quads processed (20 downloaded, 0 skipped)
2025-04-27 22:00:51,641 - INFO - Progress: 30/43 quads processed (30 downloaded, 0 skipped)
2025-04-27 22:01:03,099 - INFO - Progress: 40/43 quads processed (40 downloaded, 0 skipped)
2025-04-27 22:01:06,524 - INFO - Progress: 43/43 quads processed (43 downloaded, 0 skipped)
2025-04-27 22:01:06,526 - INFO - Download complete. Total: 43 downloaded, 0 skipped
global_monthly_2019_02_mosaic
2025-04-27 22:01:07,018 - INFO - Created 43 records
2025-04-27 22:01:19,310 - INFO - Progress: 10/43 quads processed (10 downloaded, 0 skipped)
2025-04-27 22:01:32,681 - INFO - Progress: 20/43 quads processed (20 downloaded, 0 skipped)
2025-04-27 22:01:45,288 - INFO - Progress: 30/43 quads pro

In [ ]:

mosaics

['global_monthly_2019_01_mosaic',
 'global_monthly_2019_02_mosaic',
 'global_monthly_2019_03_mosaic',
 'global_monthly_2019_04_mosaic',
 'global_monthly_2019_05_mosaic',
 'global_monthly_2019_06_mosaic',
 'global_monthly_2019_07_mosaic',
 'global_monthly_2019_08_mosaic',
 'global_monthly_2019_09_mosaic',
 'global_monthly_2019_10_mosaic',
 'global_monthly_2019_11_mosaic',
 'global_monthly_2019_12_mosaic',
 'global_monthly_2020_01_mosaic',
 'global_monthly_2020_02_mosaic',
 'global_monthly_2020_03_mosaic',
 'global_monthly_2020_04_mosaic',
 'global_monthly_2020_05_mosaic',
 'global_monthly_2020_06_mosaic',
 'global_monthly_2020_07_mosaic',
 'global_monthly_2020_08_mosaic',
 'global_monthly_2020_09_mosaic',
 'global_monthly_2020_10_mosaic',
 'global_monthly_2020_11_mosaic',
 'global_monthly_2020_12_mosaic',
 'global_monthly_2021_01_mosaic',
 'global_monthly_2021_02_mosaic',
 'global_monthly_2021_03_mosaic',
 'global_monthly_2021_04_mosaic',
 'global_monthly_2021_05_mosaic',
 'global_month

In [ ]:
counties = gpd.read_file(
    os.path.join(config.data_dir, "ca_counties", "CA_Counties.shp")
)
counties = counties.to_crs(config.geodetic_crs)
sb_county = counties.loc[counties["NAME"] == "Santa Barbara"]
bbox = [-125, 34.25, -119.0, 38.0]
sb_county = sb_county.clip(bbox)

bbox_aoi = sb_county.geometry.total_bounds

In [ ]:
# mosaic = "global_monthly_2022_12_mosaic"
# mosaic_names = ["global_monthly_2022_12_mosaic"]
mosaic_names = ['ps_biweekly_normalized_analytic_subscription_2018-12-03_2018-12-17_mosaic']

In [ ]:
counties = gpd.read_file(
    os.path.join(config.data_dir, "ca_counties", "CA_Counties.shp")
)
counties = counties.to_crs(config.geodetic_crs)
sb_county = counties.loc[counties["NAME"] == "Santa Barbara"]
bbox = [-125, 34.25, -119.0, 38.0]
sb_county = sb_county.clip(bbox)
sb_county.boundary.plot()

In [ ]:
sb_grid = data_utils.create_grid(
    sb_county,
    resolution=0.01,
    geometry_col="geometry",
    id_col="NAME",
    return_ids=True,
)
sb_grid_gdf = gpd.GeoDataFrame(
    sb_grid,
    geometry=gpd.points_from_xy(sb_grid.lon, sb_grid.lat),
    crs="EPSG:4326",
)
sb_grid_gdf.geometry = sb_grid_gdf.geometry.buffer(0.005, cap_style=3)

print(f"Shape (row, col): {sb_grid_gdf.shape}")

sb_grid_gdf.head()

In [ ]:
planet.set_mosaic(mosaic_names[0])
bbox_aoi = sb_county.geometry.total_bounds
all_items = planet.get_items(bbox_aoi)
quad_df = planet.convert_items_to_geodataframe(all_items)
quad_df = quad_df.drop_duplicates(subset="id")

quad_df = quad_df.sjoin(sb_county, predicate="intersects")
quad_df.drop(columns=["index_right"], inplace=True)
quad_df.head()

In [ ]:
all_items

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

quad_df.boundary.plot(ax=ax, edgecolor="grey", color="deepskyblue")
sb_county.plot(ax=ax, color="None", edgecolor="black")

plt.show()

In [ ]:
intersecting = gpd.sjoin(
    sb_grid_gdf.reset_index(),
    quad_df,
    how="left",
    predicate="intersects",
)

sb_grid_gdf["id"] = intersecting.groupby("index")["id"].apply(list).values

sb_grid_gdf = sb_grid_gdf[~sb_grid_gdf["id"].apply(data_utils.contains_nan)]

sb_grid_gdf = sb_grid_gdf.to_crs(config.geodetic_crs)

file_dir = os.path.join(config.basemap_dir, mosaic_names[0], "quad_ids")
os.makedirs(file_dir, exist_ok=True)

quad_fn = os.path.join(file_dir, "quad_ids.geojson")
quad_df.to_file(quad_fn, driver="GeoJSON")

grid_fn = os.path.join(file_dir, "sb_grid_gdf.parquet")
sb_grid_gdf.to_parquet(grid_fn)

In [ ]:
for mosaic in sorted(mosaic_names, reverse=True):
    print(f"Processing {mosaic}...", end="\n")
    mosaic_dir = config.basemap_dir + "/" + mosaic
    quad_dir = mosaic_dir + "/basemap_quads"
    quad_id_fp = mosaic_dir + "/quad_ids/quad_ids.geojson"

    quad_df = gpd.read_file(quad_id_fp, driver="GeoJSON")

    quad_ids_list = quad_df.id.tolist()

    os.makedirs(quad_dir, exist_ok=True)
    existing_files = os.listdir(quad_dir)

    filtered_quad_ids_list = planet_utils.filter_existing_quad_ids(
        quad_ids_list, existing_files
    )
    chunked_quad_id_list = planet_utils.chunk_list(filtered_quad_ids_list, 100)

    print(f"N Quads: {len(quad_ids_list)}")
    print(f"N Quads Remaining: {len(filtered_quad_ids_list)}")
    print(f"N Chunks: {len(chunked_quad_id_list)}")

    for i in range(len(chunked_quad_id_list)):
        print(f"Chunk: {i + 1}/{len(chunked_quad_id_list)}", end="\n")

        order_params = {
            "name": "Basemap order with geometry",
            "source_type": "basemaps",
            "order_type": "partial",
            "products": [
                {
                    "mosaic_name": mosaic,
                    "quad_ids": chunked_quad_id_list[i],
                }
            ],
        }

        order = planet.place_order(order_params)

        print("Polling for order success...", end="\n")

        planet.poll_for_success(order, loop_time=10, num_loops=240)

        results = planet.get_results(order)

        planet.download_results(
            results,
            directory=config.basemap_dir,
            overwrite=False,
            show_progress=True,
            max_retries=5,
        )

        # print("Organizing files...", end="\n")

        # planet.organize_files(
        #     base_dir=config.basemap_dir,
        #     mosaic_name=mosaic,
        #     overwrite=False,
        #     verbose=False,
        # )

        print("Done!", end="\n\n")